# PHOBERT NER VIETNAMESE NLP PROJECT

## ENIRONMENT SETUP

In [ ]:
import json
import pandas as pd
import numpy as np
from underthesea import word_tokenize
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import datasets
import evaluate
import random
import torch

## DATA INGESTION AND PREPROCESSING

### Tag Schema Standardization

In [197]:
raw_data = pd.read_csv('../data/sentences_labeled.csv')

In [2]:
PRIORITY = {"MACHINE": 1, "COMPONENT": 1, "ERROR_CODE": 1, "LOCATION": 1, "DEFECT_SYMPTOM": 2}

entity_types = ['MACHINE','COMPONENT','DEFECT_SYMPTOM','ERROR_CODE','LOCATION']

labels = [f'{k}{x}'for k in ['B-', 'I-'] for x in entity_types]
labels.insert(0, 'O')
id2string = {k: v for k, v in enumerate(labels)}
string2id = {v: k for k, v in enumerate(labels)}

## TOKENIZATION AND FEATURE ENGINEERING

### Subword Segmentation

In [198]:
entity_spans = []

for idx, row in raw_data.iterrows():
    tagged = []

    for i in entity_types:

        if pd.isna(row[i]): continue

        instances = [v.strip() for v in row[i].split(';')]

        sentence = row['sentence']

        start = 0

        for instance in instances:
            index = sentence.find(instance, start)

            index = sentence.find(instance, 0) if index < 0 else index

            start = index + len(instance)

            tagged.append((index, index + len(instance), i))

    entity_spans.append(tagged)


In [200]:
word_spans = []

for idx, row in raw_data.iterrows():
    sentence_data = []

    sentence = row['sentence']

    words = [v for v in word_tokenize(sentence)]
    
    start = 0

    for word in words:
        cleaned_word = word.replace("_", " ")

        index = sentence.find(cleaned_word, start)

        index = sentence.find(cleaned_word, 0) if index < 0 else index

        start = index + len(word)

        sentence_data.append((index, index + len(word), word))

    word_spans.append(sentence_data)


In [202]:
claimed_list = []

for i in range(len(entity_spans)):
    sorted_row = sorted(entity_spans[i], key=lambda s: (PRIORITY[s[2]], s[0]))

    words = word_spans[i]

    tags = ['O' for i in range(len(words))]

    for span in sorted_row:
        last_claimed_idx = None
        
        for j in range(len(words)):
            word = words[j]

            if word[0] < span[1] and word[1] > span[0]:
                if tags[j] == "O":
                    if last_claimed_idx is None or word[0] != last_claimed_idx + 1:
                        tags[j] = f"B-{span[2]}" 
                        last_claimed_idx = word[1]
                    elif word[0] == last_claimed_idx + 1:
                        tags[j] = f"I-{span[2]}" 
                        last_claimed_idx = word[1]

    claimed_list.append(tags)        


### Label Realignment

In [23]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=True)

In [205]:
processed_data = []

for i in range(len(word_spans)):
    words = [w[2] for w in word_spans[i]]
    tags = claimed_list[i]

    input_ids = [tokenizer.cls_token_id]
    labels = [-100]

    for word, tag in zip(words, tags):
        subword_ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(word))
        input_ids.extend(subword_ids)
        labels.append(string2id[tag])
        labels.extend([-100] * (len(subword_ids) - 1))

    input_ids.append(tokenizer.sep_token_id)
    labels.append(-100)

    processed_data.append({"input_ids": input_ids, "labels": labels})

### Processed Data Export

In [ ]:
random.seed(36)
random.shuffle(processed_data)

n = len(processed_data)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

train_data = processed_data[:n_train]
val_data = processed_data[n_train:n_train+n_val]
test_data = processed_data[n_train+n_val:]

splits = {"train": train_data, "val": val_data, "test": test_data}

for name, split in splits.items():
    with open(f"{name}.jsonl", "w", encoding="utf-8") as f:
        for row in split:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open("label_list.json", "w", encoding="utf-8") as f:
    json.dump({"string2id": string2id, "id2string": id2string}, f, ensure_ascii=False, indent=2)

## Architecture Definition And Training

### Data Loading

In [3]:
with open('../data/label_list.json', 'r', encoding='utf-8') as label_json:
    label_list = json.load(label_json)
    label_list['id2string'] = {int(k) : v for k, v in label_list['id2string'].items()}

with open('../data/train.jsonl', 'r', encoding='utf-8') as train_json:
    train = [json.loads(line) for line in train_json]
    train = datasets.Dataset.from_list(train)

with open('../data/test.jsonl', 'r', encoding='utf-8') as test_json:
    test = [json.loads(line) for line in test_json]
    test = datasets.Dataset.from_list(test)

with open('../data/val.jsonl', 'r', encoding='utf-8') as val_json:
    val = [json.loads(line) for line in val_json]
    val = datasets.Dataset.from_list(val)




### Evaluation Helper

In [4]:
def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred[0], axis=-1)
    labels = np.array(eval_pred[1])

    preds_list = []
    labels_list = []

    for i in range(len(labels)):
        sent_preds = []
        sent_labels = []
        for j in range(len(labels[i])):
            if labels[i][j] == -100:
                continue
            sent_preds.append(label_list['id2string'][preds[i][j]])
            sent_labels.append(label_list['id2string'][labels[i][j]])

        preds_list.append(sent_preds)
        labels_list.append(sent_labels)
                

    seqeval_metric = evaluate.load('seqeval')
    score = seqeval_metric.compute(predictions=preds_list, references=labels_list)

    return score
    

### Model Architecture Instantiation And Hyperparameter Configuration

In [42]:
model = AutoModelForTokenClassification.from_pretrained("vinai/phobert-base", num_labels=len(label_list['id2string']), id2label=label_list['id2string'], label2id=label_list['string2id'])
collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir = "../results",
    learning_rate = 2e-5,
    num_train_epochs = 15,
    eval_strategy = 'epoch',
    save_strategy='epoch',
    per_device_eval_batch_size = 8,
    per_device_train_batch_size = 8,
    warmup_ratio = 0.1,
    metric_for_best_model="overall_f1",
    load_best_model_at_end = True,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 36715.30it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio 

### BASELINE MODEL EVALUATION

In [ ]:
base_phobert = AutoModelForTokenClassification.from_pretrained("vinai/phobert-base", num_labels=len(label_list['id2string']), id2label=label_list['id2string'], label2id=label_list['string2id'])

In [ ]:
baseline_trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = collator,
    compute_metrics = compute_metrics,
    eval_dataset = val,
    train_dataset = train
)

In [ ]:
baseline_trainer.evaluate(test)

### Engine Handshake And Execution

In [43]:
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = collator,
    compute_metrics = compute_metrics,
    eval_dataset = val,
    train_dataset = train
)

In [44]:
trainer.train()

Epoch,Training Loss,Validation Loss,Component,Defect Type,Error Code,Location,Machine,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,No log,1.671493,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 30}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 34}",0.000000,0.000000,0.000000,0.620042
2,No log,1.374642,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 30}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 34}",0.000000,0.000000,0.000000,0.620042
3,No log,1.129382,"{'precision': 0.3333333333333333, 'recall': 0.03333333333333333, 'f1': 0.0606060606060606, 'number': 30}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.38235294117647056, 'recall': 0.38235294117647056, 'f1': 0.3823529411764706, 'number': 34}",0.378378,0.157303,0.222222,0.659708
4,No log,1.010358,"{'precision': 0.19047619047619047, 'recall': 0.13333333333333333, 'f1': 0.1568627450980392, 'number': 30}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.3333333333333333, 'recall': 0.4117647058823529, 'f1': 0.36842105263157887, 'number': 34}",0.272727,0.202247,0.232258,0.684760
5,No log,0.910406,"{'precision': 0.3103448275862069, 'recall': 0.3, 'f1': 0.3050847457627119, 'number': 30}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 8}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.3488372093023256, 'recall': 0.4411764705882353, 'f1': 0.38961038961038963, 'number': 34}",0.341463,0.314607,0.327485,0.716075
6,No log,0.824243,"{'precision': 0.25, 'recall': 0.2, 'f1': 0.22222222222222224, 'number': 30}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.8333333333333334, 'recall': 0.625, 'f1': 0.7142857142857143, 'number': 8}","{'precision': 0.3333333333333333, 'recall': 0.3333333333333333, 'f1': 0.3333333333333333, 'number': 3}","{'precision': 0.4864864864864865, 'recall': 0.5294117647058824, 'f1': 0.5070422535211269, 'number': 34}",0.375000,0.337079,0.355030,0.749478
7,No log,0.799159,"{'precision': 0.25, 'recall': 0.23333333333333334, 'f1': 0.2413793103448276, 'number': 30}","{'precision': 0.08333333333333333, 'recall': 0.07142857142857142, 'f1': 0.07692307692307691, 'number': 14}","{'precision': 0.5833333333333334, 'recall': 0.875, 'f1': 0.7000000000000001, 'number': 8}","{'precision': 0.3333333333333333, 'recall': 0.3333333333333333, 'f1': 0.3333333333333333, 'number': 3}","{'precision': 0.45454545454545453, 'recall': 0.4411764705882353, 'f1': 0.4477611940298507, 'number': 34}",0.352273,0.348315,0.350282,0.759916
8,No log,0.827814,"{'precision': 0.391304347826087, 'recall': 0.3, 'f1': 0.33962264150943394, 'number': 30}","{'precision': 0.14285714285714285, 'recall': 0.14285714285714285, 'f1': 0.14285714285714285, 'number': 14}","{'precision': 0.5833333333333334, 'recall': 0.875, 'f1': 0.7000000000000001, 'number': 8}","{'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 0.5172413793103449, 'recall': 0.4411764705882353, 'f1': 0.47619047619047616, 'number': 34}",0.432099,0.393258,0.411765,0.766180
9,No log,0.779761,"{'precision': 0.375, 'recall': 0.3, 'f1': 0.33333333333333326, 'number': 30}","{'precision': 0.17647058823529413, 'recall': 0.21428571428571427, 'f1': 0.19354

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.93it/s]
c:\miniconda3\envs\nlp\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\miniconda3\envs\nlp\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.68it/s]
c:\miniconda3\envs\nlp\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, ms

TrainOutput(global_step=375, training_loss=0.7807528483072916, metrics={'train_runtime': 62.5311, 'train_samples_per_second': 47.976, 'train_steps_per_second': 5.997, 'total_flos': 67705865238624.0, 'train_loss': 0.7807528483072916, 'epoch': 15.0})

In [45]:
trainer.evaluate(test)

Training Loss,Validation Loss,Epoch,Component,Defect Type,Error Code,Location,Machine,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
No log,0.810101,15,"{'precision': 0.48, 'recall': 0.6, 'f1': 0.5333333333333332, 'number': 20}","{'precision': 0.2, 'recall': 0.15, 'f1': 0.17142857142857143, 'number': 20}","{'precision': 0.6666666666666666, 'recall': 0.8888888888888888, 'f1': 0.761904761904762, 'number': 9}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 4}","{'precision': 0.5555555555555556, 'recall': 0.5555555555555556, 'f1': 0.5555555555555556, 'number': 27}",0.506024,0.525000,0.515337,0.763797


{'eval_loss': 0.8101014494895935,
 'eval_COMPONENT': {'precision': 0.48,
  'recall': 0.6,
  'f1': 0.5333333333333332,
  'number': 20},
 'eval_DEFECT_TYPE': {'precision': 0.2,
  'recall': 0.15,
  'f1': 0.17142857142857143,
  'number': 20},
 'eval_ERROR_CODE': {'precision': 0.6666666666666666,
  'recall': 0.8888888888888888,
  'f1': 0.761904761904762,
  'number': 9},
 'eval_LOCATION': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 4},
 'eval_MACHINE': {'precision': 0.5555555555555556,
  'recall': 0.5555555555555556,
  'f1': 0.5555555555555556,
  'number': 27},
 'eval_overall_precision': 0.5060240963855421,
 'eval_overall_recall': 0.525,
 'eval_overall_f1': 0.5153374233128835,
 'eval_overall_accuracy': 0.7637969094922737}